# A first active subspace

This tutorial finds the active subspace of the wing weight function, one of the built-in
test problems, from its gradients. It shows how to

- take a model from the built-in registry,
- sample its inputs and evaluate values and gradients,
- estimate the subspace with a bootstrap,
- choose the dimension $n$,
- read the eigenvalue, subspace-error, eigenvector and sufficient-summary plots.

The user guide pages {doc}`../user_guide/discovering` and {doc}`../user_guide/choosing_n`
give the details; {doc}`../theory/active_subspaces` and {doc}`../theory/dimension` give
the mathematics.

## The idea in two lines

Let $x$ be the inputs in normalized coordinates, with density $\rho$, and let $f$ have
gradient $\nabla f$. The matrix

$$C = \int \nabla f(x)\,\nabla f(x)^\top \rho(x)\,dx = W \Lambda W^\top,
\qquad \lambda_1 \ge \dots \ge \lambda_m \ge 0,$$

has eigenvalues $\lambda_i = \int (\nabla f^\top w_i)^2 \rho\,dx$, the mean squared
derivative of $f$ along the eigenvector $w_i$. If $\lambda_n \gg \lambda_{n+1}$, then $f$
changes mostly along the first $n$ eigenvectors $W_1$, and the **active variables**
$y = W_1^\top x$ capture most of its variation (Constantine 2015). The library estimates
$C$ by Monte Carlo, $\hat C = \frac1M \sum_{i=1}^M \nabla f(x_i)\nabla f(x_i)^\top$.

In [ ]:
%matplotlib inline
import numpy as np

import active_subspaces as ac

registry = ac.test_functions.builtin_models()
registry.names()

## The model

`registry.get` returns an `ac.Model`: a batch value function, its gradient, and an input
space that describes the inputs in **physical** units. The wing weight function
(Surjanovic and Bingham) has ten inputs, each uniform between the bounds below.

In [ ]:
wing = registry.get("wing_weight")
space = wing.space
print(wing.description, "| output:", wing.qoi_description, f"[{wing.qoi_units}]")
for name, unit, lo, hi, text in zip(
    space.names,
    space.units,
    space.lower,
    space.upper,
    wing.input_descriptions,
    strict=True,
):
    print(f"{name:>7} [{unit or '-':>8}]  {lo:>8g} .. {hi:<8g} {text}")

## Sample, evaluate, estimate

`space.sample` draws points from the input density, in physical units.
`ac.evaluate_with_grads` returns the values `f` with shape `(M,)` and the physical
gradients `G` with shape `(M, m)`. Pass both to `ac.active_subspace` with the space: the
library applies the chain rule itself to work in normalized coordinates, so do not
rescale the gradients by hand.

`nboot=200` adds 200 bootstrap replicates (resampling the rows of `G`), which the plots
use for their bands. Every random draw goes through `rng`; a fixed integer makes the
tutorial reproducible.

In [ ]:
X = space.sample(300, rng=1)
f, G = ac.evaluate_with_grads(wing, X=X)
sub = ac.active_subspace(grads=G, space=space, nboot=200, rng=2)

print("shapes:", X.shape, f.shape, G.shape)
print("chosen dimension n =", sub.n, "by rule", repr(sub.dimension_rule))
print("eigenvalues:", np.array2string(sub.eigenvalues, precision=3))

`sub` is an `ac.SubspaceResult`. It is frozen: functions that change it, such as
`ac.with_dimension` below, return a new result.

## Eigenvalues

The plotting functions return `(fig, ax)` and never show or save anything. A figure
left as the last expression of a cell is displayed; `%matplotlib inline` in the first
cell enables that.

The eigenvalue plot uses a log scale. The band is a 95% bootstrap percentile interval,
and the vertical line marks the chosen $n$.

In [ ]:
fig, ax = ac.plotting.plot_eigenvalues(sub)
fig

The same interval is available as numbers. `ac.bootstrap_interval` returns an array of
shape `(2, m)`: lower and upper limits for each eigenvalue.

In [ ]:
lo, hi = ac.bootstrap_interval(sub.bootstrap_eigenvalues, level=0.95)
for i in range(4):
    interval = f"[{lo[i]:9.3f}, {hi[i]:9.3f}]"
    print(f"lambda_{i + 1} = {sub.eigenvalues[i]:9.3f}   95% interval {interval}")

## Choosing $n$

The first eigenvalue is more than 50 times the second, so one active variable should
describe most of the variation. The default rule, `"evg"` with a linear gap, picks the
$n$ with the largest gap $\lambda_n - \lambda_{n+1}$.

The bootstrap also measures how well each subspace is determined. For each $n$,
`plot_subspace_errors` shows the distance between the estimated $W_1$ and its bootstrap
replicates, $\lVert \hat W_1 \hat W_1^\top - W_1^\ast W_1^{\ast\top} \rVert_2$, the sine
of the largest principal angle between them. A small distance at $n$ and a large gap
after $\lambda_n$ go together.

In [ ]:
fig, ax = ac.plotting.plot_subspace_errors(sub)
print("mean bootstrap distance, n = 1..4:", np.round(ac.mean_distance(sub)[:4], 3))
fig

`ac.dimension_criterion` returns the criterion a rule computes and the $n$ it picks;
`plot_dimension_criterion` draws it. Other rules are described in
{doc}`../user_guide/choosing_n`. You can always set $n$ yourself: `ac.with_dimension`
returns a copy with a new dimension and leaves `sub` unchanged.

In [ ]:
ks, gaps, n_evg = ac.dimension_criterion(sub, rule="evg")
print("EVG picks n =", n_evg)
sub2 = ac.with_dimension(sub, 2)
print("sub.n =", sub.n, " sub2.n =", sub2.n)
fig, ax = ac.plotting.plot_dimension_criterion(sub, rule="evg")
fig

## Eigenvectors

The components of $w_1$ are the weights of the normalized inputs in the active variable
$y_1 = w_1^\top x$. Their magnitudes rank the inputs by importance. The library makes
the largest-magnitude component of each eigenvector positive. The tick labels come from
`space.names`.

In [ ]:
fig, axes = ac.plotting.plot_eigenvectors(sub, indices=[0, 1])
fig

The load factor `Nz`, the aspect ratio `A`, the airfoil thickness ratio `tc` (with the
opposite sign: thicker wings weigh less here), the wing area `Sw` and the design gross
weight `Wdg` dominate; the fuel weight `Wfw` and the sweep angle `Lambda` hardly matter.

## Sufficient summary plots

A sufficient summary plot shows $f$ against the active variables. It takes the
**physical** samples `X` and computes $y = W_1^\top x$ itself. With $n = 1$ the points
lie close to a curve, so $f$ is nearly a function of $y_1$ alone.

In [ ]:
fig, ax = ac.plotting.plot_sufficient_summary(sub, X=X, f=f)
fig

With $n = 2$ the plot is a scatter of $(y_1, y_2)$ coloured by $f$. The colour changes
almost only along $y_1$.

In [ ]:
fig, ax = ac.plotting.plot_sufficient_summary(sub2, X=X, f=f)
fig

## Checking against a reference

Each engineering test function carries reference results computed once from $10^6$
samples. The estimate from 300 gradients agrees with them closely. The helper `dist` is
the projector distance used above.

In [ ]:
def dist(U, V):
    """Projector distance between the spans of U and V (sine of the largest principal angle)."""
    U = np.linalg.qr(np.reshape(U, (len(U), -1)))[0]
    V = np.linalg.qr(np.reshape(V, (len(V), -1)))[0]
    return np.linalg.norm(U @ U.T - V @ V.T, 2)

In [ ]:
ref = wing.reference_results
print("reference eigenvalues:", np.array2string(ref.eigenvalues[:3], precision=3))
print("estimated eigenvalues:", np.array2string(sub.eigenvalues[:3], precision=3))
print(f"distance between estimated and reference w_1: {dist(sub.W1, ref.eigenvectors[:, :1]):.4f}")

## Where next

- {doc}`values_only`: the same analysis when you have only values, not gradients.
- {doc}`physical_inputs`: physical and normalized coordinates, and saving results.
- {doc}`exploiting_the_subspace`: surrogates, integration and optimization on the active
  variables.

## References

- Constantine, P. G. (2015). *Active Subspaces: Emerging Ideas for Dimension Reduction
  in Parameter Studies.* SIAM.
- Constantine, P. G., Dow, E., and Wang, Q. (2014). Active subspace methods in theory
  and practice: applications to kriging surfaces. *SIAM Journal on Scientific
  Computing*, 36(4), A1500–A1524.
- Surjanovic, S., and Bingham, D. Virtual Library of Simulation Experiments: Test
  Functions and Datasets. <https://www.sfu.ca/~ssurjano/>.